# Leaflet cluster map of talk locations

This notebook extracts talk locations from `_talks/*.md`, geocodes them using OpenStreetMap (Nominatim), and generates a cluster map using `getorg`.

In [ ]:
# Install dependencies
!pip install python-frontmatter getorg geopy --quiet

import frontmatter
import glob
import getorg
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut

In [ ]:
# Collect markdown files
g = glob.glob("_talks/*.md")
print("Found files:", len(g))

In [ ]:
# Configuration
TIMEOUT = 20

# Geocoder (IMPORTANT: user_agent required)
geocoder = Nominatim(user_agent="github-actions-talkmap", timeout=TIMEOUT)

location_dict = {}
location = ""
title = ""

## Geocoding talk locations

In [ ]:
# Process each talk file
for file in g:
    data = frontmatter.load(file).to_dict()

    if 'location' not in data:
        continue

    title = data.get('title', '').strip()
    venue = data.get('venue', '').strip()
    location = data.get('location', '').strip()

    description = f"{title}<br />{venue}; {location}"

    try:
        result = geocoder.geocode(location)
        if result:
            location_dict[description] = result
            print("OK:", description)
        else:
            print("FAILED:", location)

    except GeocoderTimedOut:
        print("TIMEOUT:", location)
    except Exception as e:
        print("ERROR:", location, e)

In [ ]:
# Generate map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(
    location_dict,
    folder_name="talkmap",
    hashed_usernames=False
)

print("Map generated in /talkmap")